# Stage 3 Final Micro-Ablation Clean: v6d vs v6f (Kaggle)

Leakage-free full-val comparison between the clean v6d control and the clean v6f micro-ablation. Both variants use `_nocroppath` user templates.


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
JSONL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")

BACKEND_MODE = "qwen_hf"
RUN_ID_PREFIX = "stage3_qwen_val_v2_v6d_vs_v6f_clean"

CONTROL_VERSION = "qwen_vlm_labels_v1_prompt_v6d_balanced_notaglock_nocroppath"
CANDIDATE_VERSION = "qwen_vlm_labels_v1_prompt_v6f_balanced_notaglock_soft_ambiguous_raise_nocroppath"
PROMPT_VERSIONS = [CONTROL_VERSION, CANDIDATE_VERSION]


In [ ]:
def sh(cmd: str, cwd: Path | None = None, check: bool = True):
    print(f"$ {cmd}")
    p = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")
    return p


def require_path(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))


def metrics_row(prompt_version: str, run_id: str, run_dir: Path, eval_dir: Path) -> dict:
    metrics = read_json(eval_dir / 'metrics.json')
    rates = metrics.get('rates', {})
    f1 = metrics.get('f1', {})
    row = {
        'prompt_version': prompt_version,
        'run_id': run_id,
        'coarse_acc': float(rates.get('coarse_class_accuracy', 0.0)),
        'coarse_macro_f1': float(f1.get('coarse_class_macro_f1', 0.0)),
        'visibility_acc': float(rates.get('visibility_accuracy', 0.0)),
        'visibility_macro_f1': float(f1.get('visibility_macro_f1', 0.0)),
        'needs_review_acc': float(rates.get('needs_review_accuracy', 0.0)),
        'tag_mean_jaccard': float(rates.get('tag_mean_jaccard', 0.0)),
        'pred_ambiguous_rate': float(rates.get('pred_ambiguous_rate', 0.0)),
        'gt_ambiguous_rate': float(rates.get('gt_ambiguous_rate', 0.0)),
    }
    row['abs_ambiguous_gap'] = abs(row['pred_ambiguous_rate'] - row['gt_ambiguous_rate'])
    return row



In [ ]:
# 1) Setup: clean clone + deps once
if REPO_DIR.exists():
    print('Removing existing repo to avoid stale prompt files:', REPO_DIR)
    shutil.rmtree(REPO_DIR)

sh(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
git_head = subprocess.check_output('git rev-parse --short HEAD', shell=True, cwd=str(REPO_DIR), text=True).strip()
print('git_head:', git_head)

sh('python -m pip install -q -U pip')
sh(f"python -m pip install -q -r {REPO_DIR / 'requirements.txt'}")
sh('python -m pip install -q -U transformers accelerate qwen-vl-utils')

sh('nvidia-smi', check=False)



In [ ]:
# 2) Resolve dataset path
VAL_JSONL = None
DATASET_ROOT = None
for root in DATASET_ROOT_CANDIDATES:
    p = root / JSONL_REL
    if p.exists():
        VAL_JSONL = p
        DATASET_ROOT = root
        break

if VAL_JSONL is None:
    found = list(Path('/kaggle/input').rglob('vlm_labels_v1_val_v2.annotated.jsonl'))
    if found:
        VAL_JSONL = found[0]
        DATASET_ROOT = VAL_JSONL.parents[2]

if VAL_JSONL is None:
    raise FileNotFoundError('Could not locate vlm_labels_v1_val_v2.annotated.jsonl under /kaggle/input')

print('VAL_JSONL:', VAL_JSONL)



In [ ]:
# 3) Hard preflight checks for clean v6d/v6f registry + prompt content
import yaml

cfg_path = REPO_DIR / "configs" / "pipeline" / "stage3_vlm_gt_baseline.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
versions = cfg["prompt"]["versions"]

expected_paths = {
    CONTROL_VERSION: (
        "configs/pipeline/prompts/stage3_vlm_system_v6d_balanced_notaglock.txt",
        "configs/pipeline/prompts/stage3_vlm_user_v6d_balanced_notaglock_nocroppath.txt",
    ),
    CANDIDATE_VERSION: (
        "configs/pipeline/prompts/stage3_vlm_system_v6f_balanced_notaglock_soft_ambiguous_raise.txt",
        "configs/pipeline/prompts/stage3_vlm_user_v6f_balanced_notaglock_soft_ambiguous_raise_nocroppath.txt",
    ),
}

positive_sentence = 'If the visible evidence itself cannot be trusted due to blur, glare, washout, low contrast, heavy shadow, or unstable boundaries, prefer visibility="ambiguous" even when a conservative coarse_class guess is still possible.'
anti_sentence = 'Do not use visibility="ambiguous" only because defect evidence is weak, absent, or conservative while the visible region remains readable.'
soft_raise_sentence = 'If the visible evidence is borderline readable and a suspicious mark or boundary could plausibly be explained by image quality rather than object structure, prefer visibility="ambiguous" over visibility="clear".'
taglock_line = 'If visibility="ambiguous", ensure the tags explain why.'

for pv in PROMPT_VERSIONS:
    if pv not in versions:
        raise RuntimeError(f"Prompt version missing in config: {pv}")
    expected_sys, expected_usr = expected_paths[pv]
    actual_sys = versions[pv]["system_path"]
    actual_usr = versions[pv]["user_path"]
    if actual_sys != expected_sys or actual_usr != expected_usr:
        raise RuntimeError(
            f"Registry mismatch for {pv}: expected ({expected_sys}, {expected_usr}), got ({actual_sys}, {actual_usr})"
        )

    sys_abs = REPO_DIR / actual_sys
    usr_abs = REPO_DIR / actual_usr
    require_path(sys_abs, f"system prompt for {pv}")
    require_path(usr_abs, f"user prompt for {pv}")
    sys_text = sys_abs.read_text(encoding="utf-8")
    usr_text = usr_abs.read_text(encoding="utf-8")
    if "crop_path" in usr_text:
        raise RuntimeError(f"Clean user prompt still exposes crop_path: {pv}")
    if taglock_line in usr_text:
        raise RuntimeError(f"Unexpected tag-lock line in clean notaglock prompt: {pv}")
    if positive_sentence not in sys_text or anti_sentence not in sys_text:
        raise RuntimeError(f"Missing balanced visibility rules in {pv}")
    if pv == CANDIDATE_VERSION and soft_raise_sentence not in sys_text:
        raise RuntimeError("Missing soft ambiguous raise sentence in clean v6f")
    if pv == CONTROL_VERSION and soft_raise_sentence in sys_text:
        raise RuntimeError("Soft ambiguous raise sentence leaked into clean v6d control")

print("Clean v6d/v6f prompt checks passed.")


In [ ]:
# 4) Run full-val for exactly two versions
rows = []
run_dirs = {}

for pv in PROMPT_VERSIONS:
    run_id = f"{RUN_ID_PREFIX}_{pv}"
    print("\n" + "=" * 88)
    print('Running:', pv)

    sh(
        " ".join([
            'python',
            str(REPO_DIR / 'scripts' / 'run_stage3_vlm_baseline.py'),
            '--config', str(REPO_DIR / 'configs' / 'pipeline' / 'stage3_vlm_gt_baseline.yaml'),
            '--backend-mode', BACKEND_MODE,
            '--prompt-version', pv,
            '--input-jsonl', f'"{VAL_JSONL}"',
            '--run-id', run_id,
            '--no-resume',
        ]),
        cwd=REPO_DIR,
    )

    run_dir = REPO_DIR / 'outputs' / 'stage3_vlm_baseline_runs' / run_id
    eval_dir = run_dir / 'eval'
    pred_jsonl = run_dir / 'predictions_vlm_labels_v1.jsonl'

    require_path(run_dir, f'run_dir {pv}')
    require_path(pred_jsonl, f'predictions {pv}')

    sh(
        " ".join([
            'python',
            str(REPO_DIR / 'scripts' / 'validate_vlm_labels_v1.py'),
            '--input', f'"{pred_jsonl}"',
        ]),
        cwd=REPO_DIR,
    )

    sh(
        " ".join([
            'python',
            str(REPO_DIR / 'scripts' / 'eval_stage3_vlm_baseline.py'),
            '--run-dir', f'"{run_dir}"',
            '--ground-truth-jsonl', f'"{VAL_JSONL}"',
        ]),
        cwd=REPO_DIR,
    )

    sh(
        " ".join([
            'python',
            str(REPO_DIR / 'scripts' / 'visualize_stage3_eval_results.py'),
            '--eval-dir', f'"{eval_dir}"',
        ]),
        cwd=REPO_DIR,
    )

    rows.append(metrics_row(prompt_version=pv, run_id=run_id, run_dir=run_dir, eval_dir=eval_dir))
    run_dirs[pv] = run_dir

print('Completed runs:', len(rows))



In [ ]:
# 5) Compact comparison table + decision rule
if len(rows) != 2:
    raise RuntimeError(f'Expected 2 rows, got {len(rows)}')

df = pd.DataFrame(rows)
order_cols = [
    'prompt_version',
    'coarse_acc',
    'coarse_macro_f1',
    'visibility_acc',
    'visibility_macro_f1',
    'needs_review_acc',
    'tag_mean_jaccard',
    'pred_ambiguous_rate',
    'gt_ambiguous_rate',
    'abs_ambiguous_gap',
]
comparison = df[order_cols].copy()

row_v6d = comparison[comparison['prompt_version'] == CONTROL_VERSION].iloc[0]
row_v6f = comparison[comparison['prompt_version'] == CANDIDATE_VERSION].iloc[0]

select_v6f = (
    row_v6f['visibility_macro_f1'] >= row_v6d['visibility_macro_f1']
    and row_v6f['abs_ambiguous_gap'] < row_v6d['abs_ambiguous_gap']
    and row_v6f['coarse_acc'] >= row_v6d['coarse_acc']
    and row_v6f['pred_ambiguous_rate'] <= 0.15
)

selected = CANDIDATE_VERSION if select_v6f else CONTROL_VERSION
recommendation = 'replace_with_v6f' if select_v6f else 'freeze_v6d_stop_prompt_tuning'

print('Selected prompt:', selected)
print('Recommendation:', recommendation)

display(comparison)

AGG_DIR = REPO_DIR / 'outputs' / 'stage3_vlm_baseline_runs' / f'{RUN_ID_PREFIX}_aggregate'
AGG_DIR.mkdir(parents=True, exist_ok=True)

csv_path = AGG_DIR / 'v6d_vs_v6f_comparison.csv'
md_path = AGG_DIR / 'v6d_vs_v6f_comparison.md'
comparison.to_csv(csv_path, index=False)

def md_table(df_in: pd.DataFrame) -> str:
    cols = list(df_in.columns)
    lines = []
    lines.append('| ' + ' | '.join(cols) + ' |')
    lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
    for _, rr in df_in.iterrows():
        vals = []
        for c in cols:
            v = rr[c]
            if isinstance(v, float):
                vals.append(f'{v:.6f}')
            else:
                vals.append(str(v))
        lines.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join(lines)

md_lines = [
    '# v6d vs v6f Comparison',
    '',
    f'- selected_prompt: `{selected}`',
    f'- recommendation: `{recommendation}`',
    '',
    md_table(comparison),
]
md_path.write_text('\n'.join(md_lines), encoding='utf-8')

decision_json = AGG_DIR / 'selection_decision.json'
decision_json.write_text(json.dumps({
    'selected_prompt': selected,
    'recommendation': recommendation,
    'rule': {
        'visibility_macro_f1_ge_v6d': bool(row_v6f['visibility_macro_f1'] >= row_v6d['visibility_macro_f1']),
        'abs_ambiguous_gap_lt_v6d': bool(row_v6f['abs_ambiguous_gap'] < row_v6d['abs_ambiguous_gap']),
        'coarse_acc_no_regress': bool(row_v6f['coarse_acc'] >= row_v6d['coarse_acc']),
        'pred_ambiguous_rate_le_0_15': bool(row_v6f['pred_ambiguous_rate'] <= 0.15),
    },
}, ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved:', csv_path)
print('Saved:', md_path)
print('Saved:', decision_json)



In [ ]:
# 6) Package deliverables
DELIVER_DIR = Path(f"/kaggle/working/stage3_deliverables_{RUN_ID_PREFIX}")
if DELIVER_DIR.exists():
    shutil.rmtree(DELIVER_DIR)
DELIVER_DIR.mkdir(parents=True, exist_ok=True)

aggregate_files = [AGG_DIR / 'v6d_vs_v6f_comparison.csv', AGG_DIR / 'v6d_vs_v6f_comparison.md']
for p in aggregate_files:
    if p.exists():
        dst = DELIVER_DIR / "aggregate" / p.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dst)

run_artifacts_rel = [
    "run_summary.json",
    "config_snapshot.json",
    "predictions_vlm_labels_v1.jsonl",
    "parsed_predictions.jsonl",
    "raw_responses.jsonl",
    "sample_results.jsonl",
    "failures.jsonl",
    "eval/metrics.json",
    "eval/confusion_coarse_class.csv",
    "eval/confusion_visibility.csv",
    "eval/review_table.csv",
    "eval/failures.jsonl",
    "eval/visuals/report.md",
    "eval/visuals/kpi_overview.png",
    "eval/visuals/confusion_coarse_class.png",
    "eval/visuals/confusion_visibility.png",
    "eval/visuals/visibility_errors_top.csv",
    "eval/visuals/visibility_errors_gallery.png",
]

for pv in PROMPT_VERSIONS:
    run_dir = run_dirs[pv]
    for rel in run_artifacts_rel:
        src_path = run_dir / rel
        if src_path.exists():
            dst = DELIVER_DIR / "runs" / pv / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_path, dst)

summary_md = DELIVER_DIR / "RESULT_SUMMARY.md"
summary_md.write_text(
    "\n".join([
        f"# Stage 3 Clean Micro-Ablation Result: {RUN_ID_PREFIX}",
        "",
        f"- control_version: {CONTROL_VERSION}",
        f"- candidate_version: {CANDIDATE_VERSION}",
        f"- selected_version: {selected_version}",
        f"- decision: {decision}",
        "",
        f"Ground truth JSONL: {VAL_JSONL}",
        f"Repo commit: {git_head}",
    ]),
    encoding="utf-8",
)

ARCHIVE_BASE = Path(f"/kaggle/working/stage3_deliverables_{RUN_ID_PREFIX}")
ARCHIVE_PATH = shutil.make_archive(str(ARCHIVE_BASE), "gztar", root_dir=DELIVER_DIR)

print("DELIVER_DIR:", DELIVER_DIR)
print("ARCHIVE_PATH:", ARCHIVE_PATH)


## Artifacts

Per-run outputs: `outputs/stage3_vlm_baseline_runs/<RUN_ID_PREFIX>...`
Packed archive: `/kaggle/working/stage3_deliverables_stage3_qwen_val_v2_v6d_vs_v6f_clean.tar.gz`
